In [15]:
import os
import re
import unicodedata
import glob
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
import nltk
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize
from stop_words_custom import stop_word_list
from unpack_documents import load_documents, preprocess, CustomPorterStemmer

# Directory containing .txt files
TEXT_DIR = "text" 

# Directory containing other metadata
DATA_DIR = "data"

In [2]:
male_pronouns = ["he", "him"]
female_pronouns = ["she", "her"]

## Load documents

In [3]:
filepaths = glob.glob(os.path.join(TEXT_DIR, "*.txt"))

corpus, filenames, titles = load_documents(filepaths)

Loaded 762 documents


## Preprocessing documents (stemming)

In [4]:
stemmer = CustomPorterStemmer()

corpus_stemmed = [preprocess(doc, stemmer) for doc in corpus]

## Bag of words

In [5]:
vectorizer = CountVectorizer(
    strip_accents='unicode',
    stop_words=stop_word_list,
    max_df=1.0,
    min_df=5
)

X = vectorizer.fit_transform(corpus_stemmed)

print("Matrix shape:", X.shape)

/Users/mjl09005/Library/CloudStorage/OneDrive-UniversityofConnecticut/gvp/docx_env/lib/python3.11/site-packages/sklearn/feature_extraction/text.py:411: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['articl', 'mon'] not in stop_words.
  warnings.warn(


Matrix shape: (762, 14355)


In [6]:
feature_names = vectorizer.get_feature_names_out()

bow_df = pd.DataFrame(X.toarray(), columns=feature_names, index=titles)

bow_df.head()

,aa,aaron,aback,abalon,abandon,abash,abat,abattoir,abbey,abbi,...,zigzag,zillion,zinc,zip,zipper,zombi,zone,zoo,zoom,zu
Brisk Money,0,0,0,0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0
Specimen 313,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
When We Were Heroes,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
Tie A Yellow Ribbon,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Apologue,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,1,0,0,0


In [7]:
word_counts = bow_df.sum(axis=0).sort_values(ascending=False)

word_counts.head()

he      70200
him     69431
she     59722
her     58112
they    29427
dtype: int64

In [8]:
sum_male = 0
sum_female = 0

for pronoun in male_pronouns:
    sum_male += word_counts.get(pronoun, 0)

for pronoun in female_pronouns:
    sum_female += word_counts.get(pronoun, 0)

print(f"Total male pronouns: {sum_male}")
print(f"Total female pronouns: {sum_female}")

Total male pronouns: 139631
Total female pronouns: 117834


In [9]:
bow_df["male_total"] = bow_df[male_pronouns].sum(axis=1)
bow_df["female_total"] = bow_df[female_pronouns].sum(axis=1)

In [32]:
stats = pd.read_csv('data/compiled_stats.csv')

In [33]:
stats['cleaned_title'] = (
       stats['Title']
       #.apply(lambda x: unicodedata.normalize('NFKD', x).encode('ascii', 'ignore').#decode('ascii')
       #          if isinstance(x, str) else x)
       .str.replace(r"[:#/]", "", regex=True)
)

In [34]:
pronoun_data = (
    stats[stats['Type']=="Original Fiction"].merge(
        bow_df[['male_total', 'female_total']],
        left_on='cleaned_title',
        right_index=True,
        how='left'
    )
)

In [35]:
pronoun_data.head()

,Publication Date,Publication Year,Title,Author,Pronoun,Edited by,Edited by 2,Series,Type,Category,Subcategory,Subsubcategory,Word Count,Page Count,cleaned_title,male_total,female_total
1,20-Jul-08,2008,Down on the Farm,Charles Stross,He/him,Patrick Nielsen Hayden,NaN,NaN,Original Fiction,Fantasy,Lovecraftian,NaN,12682.0,NaN,Down on the Farm,159.0,113.0
2,20-Jul-08,2008,After the Coup,John Scalzi,He/him,Patrick Nielsen Hayden,NaN,NaN,Original Fiction,Science Fiction,Space Opera,NaN,7087.0,NaN,After the Coup,237.0,12.0
4,6-Aug-08,2008,The Things That Make Me Weak and Strange Get E...,Cory Doctorow,He/him,Patrick Nielsen Hayden,NaN,NaN,Original Fiction,NaN,NaN,NaN,15708.0,NaN,The Things That Make Me Weak and Strange Get E...,805.0,137.0
5,26-Aug-08,2008,Shade,Steven Gould,He/him,NaN,NaN,NaN,Original Fiction,NaN,NaN,NaN,5619.0,NaN,Shade,251.0,104.0
6,11-Sep-08,2008,The Girl Who Sang Rose Madder,Elizabeth Bear,She/her,Patrick Nielsen Hayden,NaN,NaN,Original Fiction,Fantasy,NaN,NaN,5706.0,NaN,The Girl Who Sang Rose Madder,123.0,315.0


In [36]:
num_nans = pronoun_data['female_total'].isna().sum()
print(num_nans)

49


In [37]:
pronoun_data[pronoun_data['female_total'].isna()]

,Publication Date,Publication Year,Title,Author,Pronoun,Edited by,Edited by 2,Series,Type,Category,Subcategory,Subsubcategory,Word Count,Page Count,cleaned_title,male_total,female_total
52,4-Dec-09,2009,I Speak Fluent Giraffe,Jason Henninger,He/him,NaN,NaN,NaN,Original Fiction,Humor,NaN,NaN,0.0,NaN,I Speak Fluent Giraffe,NaN,NaN
71,21-Apr-10,2010,"Four Horsemen, at Their Leisure",Richard Parks,He/him,Patrick Nielsen Hayden,NaN,NaN,Original Fiction,Apocalyptic and Post-Apocalyptic,Fantasy,NaN,3648.0,NaN,"Four Horsemen, at Their Leisure",NaN,NaN
83,12-Jul-10,2010,The President's Brain Is Missing,John Scalzi,He/him,Patrick Nielsen Hayden,NaN,NaN,Original Fiction,Humor,Science Fiction,NaN,6695.0,NaN,The President's Brain Is Missing,NaN,NaN
121,21-Dec-10,2010,The Trains that Climb the Winter Tree,Michael Swanwick,He/him,NaN,NaN,NaN,Original Fiction,Fantasy,NaN,NaN,11556.0,NaN,The Trains that Climb the Winter Tree,NaN,NaN
122,21-Dec-10,2010,The Trains that Climb the Winter Tree,Eileen Gunn,She/her,NaN,NaN,NaN,Original Fiction,Fantasy,NaN,NaN,11556.0,NaN,The Trains that Climb the Winter Tree,NaN,NaN
130,1-Apr-11,2011,"The Shadow War of the Night Dragons, Book One:...",John Scalzi,He/him,NaN,NaN,NaN,Original Fiction,Book Excerpt,NaN,NaN,3741.0,NaN,"The Shadow War of the Night Dragons, Book One ...",NaN,NaN
226,6-Jun-12,2012,The Witch of Duva: A Ravkan Folk Tale,Leigh Bardugo,She/her,Noa Wheeler,NaN,NaN,Original Fiction,Fantasy,Young Adult,NaN,7834.0,NaN,The Witch of Duva A Ravkan Folk Tale,NaN,NaN
276,12-Mar-13,2013,Border Dogs: A SEAL Team 666 Adventure,Weston Ochse,He/him,Brendan Deneen,NaN,NaN,Original Fiction,Fantasy,Urban Fantasy,NaN,11735.0,NaN,Border Dogs A SEAL Team 666 Adventure,NaN,NaN
342,8-Oct-13,2013,Slayers: The Making of a Mentor,C. J. Hill,She/her,Lauren Burniac,NaN,NaN,Original Fiction,Fantasy,Young Adult,NaN,16910.0,NaN,Slayers The Making of a Mentor,NaN,NaN
377,26-Mar-14,2014,Anyway: Angie,Daniel José Older,He/him,Carl Engle-Laird,NaN,NaN,Original Fiction,Horror,Urban Fantasy,NaN,6950.0,NaN,Anyway Angie,NaN,NaN
